# Polymorphism in Python



## 1. Introduction
Polymorphism means **one name, many forms**. In object-oriented programming, it allows us to use the **same method name** or **same interface** for different kinds of objects.

In this notebook, we will walk through several forms of polymorphism in Python:

1. Same method name, different classes
2. Polymorphism through inheritance (method overriding)
3. Runtime (dynamic) polymorphism
4. Polymorphism with abstract base classes
5. Operator overloading (special methods)
6. Multiple inheritance and method resolution order (MRO)
7. A mini project that combines all of the above



## 2. Same Method Name, Different Classes

We start with the simplest and most intuitive example: different classes that define the same method.


In [1]:
class Dog:
    def speak(self):
        return "Woof! I am a dog"

class Cat:
    def speak(self):
        return "Meow! I am a cat"

In [2]:
dog = Dog()
cat = Cat()

In [3]:
dog.speak()

'Woof! I am a dog'

In [4]:
cat.speak()

'Meow! I am a cat'


**Explanation:**  
- Both `Dog` and `Cat` have a method called `speak()`.
- When we call `speak()` on a `Dog`, we get dog-like behavior.
- When we call `speak()` on a `Cat`, we get cat-like behavior.
- The **name** of the method is the same, but **the behavior depends on the object** → this is polymorphism.


In [6]:
dog2 = Dog()
cat2 = Cat()
animals = [dog,cat,dog2,cat2]

for animal in animals:
    print(animal.speak())

Woof! I am a dog
Meow! I am a cat
Woof! I am a dog
Meow! I am a cat



This already shows a key idea: **we can write code that works on a collection of different objects, as long as they all provide the same method.**



## 3. Polymorphism Through Inheritance (Method Overriding)

Now let's connect this to inheritance. A parent (base) class defines a method, and child (derived) classes **override** that method to give it more specific behavior.


In [9]:
class Bird:
    def fly(self):
        return "This bird flies in a general way"

class Parrot(Bird):
    def fly(self):
        return "The parrot flies quickly"

class Penguin(Bird):
    def fly(self):
        return "Penguin cannot fly, but they swim instead"

        

In [11]:
birds = [Bird(),Parrot(),Penguin()]
for bird in birds:
    print(bird.fly())

This bird flies in a general way
The parrot flies quickly
Penguin cannot fly, but they swim instead



Here, all objects respond to `fly()`, but each in their **own** way. This is **runtime polymorphism**: the method that runs is chosen when the program is running, based on the actual object.



### What if a subclass does **not** override the method?
If a child class does **not** define the method, Python will look it up in the parent class.


In [15]:
class Sparrow(Bird):
    pass
class Eagle(Bird):
    def fly(self):
        return "THe eagle flies high above the mountain"

In [16]:
sparrow = Sparrow()
eagle = Eagle()

In [17]:
print(sparrow.fly())
print(eagle.fly())

This bird flies in a general way
THe eagle flies high above the mountain



This shows the inheritance lookup: Python first looks in the child; if it doesn't find the method, it goes to the parent.



## 4. Runtime (Dynamic) Polymorphism

**Runtime polymorphism** means: *the method that actually gets executed is determined while the program is running, not when we write the code.*

When we write:

```python
for b in birds:
    b.fly()
```

We don't know ahead of time which exact class `b` is. At **runtime**, Python checks the actual object (`Bird`, `Parrot`, or `Penguin`) and then calls the matching `fly()`.



## 5. Polymorphism with Abstract Base Classes

Sometimes we want to **force** all subclasses to provide a certain method. We can do this with *abstract base classes* using the `abc` module.


In [20]:
class Rectangle:
    def area(self):
        return 10*5

class Circle:
    def area(self):
        return 3.14 * 3**2

shapes = [Rectangle(),Circle()]
for s in shapes:
    print(s.area())
    

50
28.26


In [21]:
# show the problem
class Triangle:
    pass
t = Triangle()
t.area()

#Problem :  we have no guarantee that all shapes implement area()

AttributeError: 'Triangle' object has no attribute 'area'

In [24]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self):
        pass

In [25]:
class Rectangle(Shape):
    def __init__(self,w,h):
        self.w = w
        self.h = h
    def area(self):
        return self.w * self.h

class Circle(Shape):
    def __init__(self,r):
        self.r = r
    def area(self):
        return 3.14 * self.r^2

    


If we try to create a subclass of `Shape` without implementing `area()`, Python will not let us instantiate it; this helps us keep a **consistent polymorphic interface**.



## 6. Operator Overloading (Special Methods)

Python lets us define how operators like `+`, `-`, or even `print()` should work for our own classes.  
This is another form of polymorphism: **the same operator behaves differently depending on the object type.**


In [29]:
class Vector:
    def __init__(self,x,y):
        self.x = x
        self.y = y
    def __add__(self,other):
        return Vector(self.x + other.x,self.y + other.y)  
    def __str__(self):
        return str(self.x) + " " + str(self.y)

In [32]:
v1 = Vector(2,3)
v2 = Vector(4,5)
v3 = v1 + v2
print(v3)

6 8



Here the same operator `+` works on integers, strings, lists, and now **our own class**; that's polymorphism.



## 7. Multiple Inheritance and Polymorphism

Python allows a class to inherit from **more than one** parent.  
If two parents define the **same method name**, Python has to decide **which one to use**.  
It does this using the **Method Resolution Order (MRO)**.


In [33]:
class A:
    def show(self):
        print("show()from class A")

class B:
    def show(self):
        print("show()from class B")

class C(A,B):
    pass

c = C()
c.show()

show()from class A



Because `C` inherits from `A` first, Python will call `A.show()`.  
We can check the MRO to see the order Python searches for methods.


In [34]:
C.__mro__

(__main__.C, __main__.A, __main__.B, object)


If we reverse the order of inheritance, we change which method is chosen.


In [36]:
class C2(B,A):
    pass
c2 = C2()
c2.show()

show()from class B



We can also override the method in the child itself — that one always wins.


In [37]:
class D(A,B):
    def show(self):
        print("show() from class D")
d = D()
d.show()

show() from class D



## 8. Mini Project — Vehicle Simulation

We want to build a small object-oriented program that works with different types of vehicles (car, boat, plane, amphibious vehicle).
Even though they are different, we want to treat them the same way in our code; that’s where polymorphism comes in.
This mini project puts together:
So the goal is:
- define a common interface (move())
- make different classes implement it differently
- show multiple inheritance
- show operator overloading in a related class

### Step 1 — Define the Common Interface (Abstract Base Class)
We need all vehicles to have:
- a `name`
- a `speed`
- a `move()` method (but we don’t know how each moves yet)

We'll make an **abstract base class** for this.


In [38]:
from abc import ABC, abstractmethod

class Vehicle(ABC):
    def __init__(self,name,speed):
        self.name = name
        self.speed = speed

    @abstractmethod
    def move(self):
        pass



> `Vehicle` is just a template. You cannot create an instance of it directly — it only defines what every vehicle should have.

### Step 2 — Create Concrete Subclasses (Runtime Polymorphism)

Each subclass must implement `move()` differently. This is **method overriding**.


In [73]:
class Car(Vehicle):
    def move(self):
        return self.name + " is driving on the road at " + str(self.speed) + "mph"

class Boat(Vehicle):
    def move(self):
        return self.name + " is sailing on the water at " + str(self.speed) + "mph"

class Plane(Vehicle):
    def move(self):
        return self.name + " is flying in the sky at " + str(self.speed) + "mph"

In [74]:
vehicles = [
    Car("Toyota",60),
    Boat("Yamaha",40),
    Plane("Boeing",500)
]

for v in vehicles:
    print(v.move())


Toyota is driving on the road at 60mph
Yamaha is sailing on the water at 40mph
Boeing is flying in the sky at 500mph


The loop doesn’t care what kind of object `v` is. Python decides **at runtime** which `move()` version to call.


### Step 3 — Add Multiple Inheritance (Two Parents)
Now let’s create a vehicle that can do **two things**: drive like a car *and* sail like a boat.


In [75]:
class Jbond007(Car,Boat):
    def move(self):
        return self.name + " can drive on land and sail on ocean at " + str(self.speed) + " mph"

In [76]:
jb007 = Jbond007("Aston-Martin",1)
jb007.move()

'Aston-Martin can drive on land and sail on ocean at 1 mph'

In [77]:
Jbond007.__mro__

(__main__.Jbond007,
 __main__.Car,
 __main__.Boat,
 __main__.Vehicle,
 abc.ABC,
 object)

Python uses the **Method Resolution Order (MRO)** to decide which parent’s method to use if there’s a conflict.


### Step 4 — Operator Overloading (Another Kind of Polymorphism)
We'll add a helper class `Trip` that represents travel distance.  
We’ll make the `+` operator combine two trips together.


In [90]:
class Trip:
    def __init__(self,distance):
        self.distance = distance

    def __add__(self,other):
        return Trip(self.distance + other.distance)

    def __sub__(self,other):
        return Trip(self.distance - other.distance)
        
    def __str__(self):
        return "Total distance is " + str(self.distance)

In [91]:
trip1 = Trip(120) 
trip2 = Trip(80)
trip3 = trip1 + trip2

In [92]:
print(trip3)

Total distance is 200


The same operator `+` works differently for integers, strings, and our new class.  
That’s polymorphism through **operator overloading**.

### Step 5 — Combine Everything

1. Create `Vehicle` (abstract) with `name`, `speed`, and abstract `move()`.
2. Implement at least three subclasses (`Car`, `Boat`, `Plane`) that override `move()`.
3. Create a `AmphibiousVehicle` inheriting from two parents.
4. Implement a `Trip` class with operator `+` support.
5. Write a test that:
   - loops through vehicles and calls `move()`,
   - adds trips,
   - prints the MRO.

In [93]:
vehicles = [
    Car("Toyota",60),
    Boat("Yamaha",40),
    Plane("Boeing",500),
    Jbond007('Aston-Martin', -50)
]

for v in vehicles:
    print(v.move())

Toyota is driving on the road at 60mph
Yamaha is sailing on the water at 40mph
Boeing is flying in the sky at 500mph
Aston-Martin can drive on land and sail on ocean at -50 mph


In [94]:
trip1 = Trip(120) 
trip2 = Trip(80)
trip3 = trip1 + trip2
print(trip1 + trip2)

Total distance is 200



### What this mini project demonstrates
- All vehicles share the same interface (`move()`), enforced by an abstract base class → **polymorphism**
- Each subclass provides its **own** version of `move()` → **runtime polymorphism / method overriding**
- `AmphibiousVehicle` shows **multiple inheritance** and its own overriding
- `Trip` shows **operator overloading**, another form of polymorphism
